# 25 — DLHA su NSL-KDD (T43–T44)

Reimplementazione di Wisanwanichthan e Thammawichai, IEEE Access 2021,
[DOI 10.1109/ACCESS.2021.3118573](https://doi.org/10.1109/ACCESS.2021.3118573).
L’algoritmo 1 produce un allarme binario. Gaussian Naive Bayes cerca DoS e
Probe. Le connessioni per cui non scatta l’allarme passano alla SVM, addestrata
su R2L, U2R e connessioni normali. Confrontiamo la cascata con le predizioni
già salvate del MLP, convertite in allarmi binari.

[Protocollo e scelte di replica](../docs/DLHA_OFFICIAL_COMPARISON.md).

## 1. Leggere e verificare i file del benchmark

In [ ]:
from pathlib import Path
import hashlib
import json
import sys

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from IPython.display import display
from sklearn.compose import ColumnTransformer
from sklearn.decomposition import PCA
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay
from sklearn.naive_bayes import GaussianNB
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import MinMaxScaler, OneHotEncoder
from sklearn.svm import SVC

ROOT = Path.cwd()
if ROOT.name == 'notebooks':
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT))
from src.data import COLUMN_NAMES, FEATURE_COLUMNS, CATEGORICAL_COLUMNS
from src.targets import ATTACK_TO_FAMILY, AUTHOR_ATTACK_TO_FAMILY

protocol_path = ROOT / 'configs/official_test_v1.json'
protocol = json.loads(protocol_path.read_text(encoding='utf-8'))
for name, digest in protocol['input_sha256'].items():
    assert hashlib.sha256((ROOT / name).read_bytes()).hexdigest() == digest

train = pd.read_csv(ROOT / 'data/raw/KDDTrain+.csv', header=None, names=COLUMN_NAMES)
test = pd.read_csv(ROOT / 'data/raw/KDDTest+.csv', header=None, names=COLUMN_NAMES)
X_train = train[list(FEATURE_COLUMNS)]
X_test = test[list(FEATURE_COLUMNS)]
train_family = train['attack_label'].map(AUTHOR_ATTACK_TO_FAMILY)
test_family = test['attack_label'].map(AUTHOR_ATTACK_TO_FAMILY)
assert train_family.notna().all() and test_family.notna().all()
assert len(train) == 125_973 and len(test) == 22_544

## 2. Preparare i due gruppi di addestramento

Usiamo le liste di feature numeriche pubblicate nella sezione V-B. Conserviamo
le tre categoriche, codificate one-hot. L’equazione 4 dà una scala [0,1],
coerente con il numero di componenti riportato nel paper. Ogni trasformazione
impara solo dal proprio gruppo di training e precede il campionamento dei normali.

In [ ]:
nb_positions = [12, 23, 25, 27, 28, 29, 30, 31, 33, 34, 35, 36, 37, 38, 39, 40, 41]
svm_positions = [9, 10, 12, 14, 18, 24, 31, 32, 33, 36, 37]
nb_columns = [FEATURE_COLUMNS[i - 1] for i in nb_positions]
svm_columns = [FEATURE_COLUMNS[i - 1] for i in svm_positions]

nb_target = train_family.isin(['DoS', 'Probe'])
svm_rows = train_family.isin(['Normal', 'R2L', 'U2R'])
svm_train = X_train.loc[svm_rows]
svm_target = train_family.loc[svm_rows].ne('Normal').to_numpy()
svm_attacks = np.flatnonzero(svm_target)
svm_normals = np.flatnonzero(~svm_target)
assert nb_target.sum() == 57_583
assert len(svm_attacks) == 1_047 and len(svm_normals) == 67_343

nb_transform = make_pipeline(ColumnTransformer([
    ('numeric', MinMaxScaler(), nb_columns),
    ('categorical', OneHotEncoder(handle_unknown='ignore', sparse_output=False), list(CATEGORICAL_COLUMNS)),
]), PCA(n_components=0.95, svd_solver='full'))
svm_transform = make_pipeline(ColumnTransformer([
    ('numeric', MinMaxScaler(), svm_columns),
    ('categorical', OneHotEncoder(handle_unknown='ignore', sparse_output=False), list(CATEGORICAL_COLUMNS)),
]), PCA(n_components=0.95, svd_solver='full'))

nb_features = nb_transform.fit_transform(X_train)
svm_features = svm_transform.fit_transform(svm_train)
assert (nb_features.shape[1], svm_features.shape[1]) == (28, 13)
display(pd.DataFrame({
    'stadio': ['Naive Bayes', 'SVM'],
    'righe per le trasformazioni': [len(nb_features), len(svm_features)],
    'componenti PCA': [nb_features.shape[1], svm_features.shape[1]],
    'varianza conservata': [p[-1].explained_variance_ratio_.sum() for p in (nb_transform, svm_transform)],
}).round(4))

## 3. Addestrare i classificatori e valutare la cascata

Per la SVM usiamo il kernel RBF e i parametri pubblicati per KDDTrain+.
Ripetiamo solo il campionamento dei 1.047 normali, con cinque seed fissati
prima del test. Il Naive Bayes viene addestrato una volta. Una connessione
già segnalata dal primo stadio mantiene il suo allarme.

In [ ]:
naive_bayes = GaussianNB().fit(nb_features, nb_target)
first_stage = naive_bayes.predict(nb_transform.transform(X_test))
to_svm = ~first_stage
svm_test = svm_transform.transform(X_test.loc[to_svm])
seeds = [1, 2, 3, 4, 5]
predictions = []
sampled_indices = []
for seed in seeds:
    normal_sample = np.random.default_rng(seed).choice(svm_normals, size=len(svm_attacks), replace=False)
    sample = np.sort(np.concatenate([svm_attacks, normal_sample]))
    svm = SVC(kernel='rbf', C=0.1, gamma=0.01)
    svm.fit(svm_features[sample], svm_target[sample])
    alarms = first_stage.copy()
    alarms[to_svm] = svm.predict(svm_test)
    predictions.append(alarms)
    sampled_indices.append(svm_train.index.to_numpy()[normal_sample])

print(f'Connessioni passate alla SVM: {int(to_svm.sum()):,}/{len(test):,}')

## 4. Confrontare rilevamento e falsi allarmi

Il confronto usa 3.750 attacchi di tipi assenti, 9.083 di tipi già presenti e
9.711 connessioni normali. Il falso allarme è un Normal segnalato come attacco.
Per il MLP qualsiasi famiglia diversa da Normal conta come un allarme.
La media DLHA descrive la sensibilità al campionamento su questo stesso test.

In [ ]:
mlp_dir = ROOT / 'artifacts/neural_network_official_test'
mlp_info = json.loads((mlp_dir / 'results.json').read_text(encoding='utf-8'))
assert mlp_info['protocol_sha256'] == hashlib.sha256(protocol_path.read_bytes()).hexdigest()
saved_path = mlp_dir / 'predictions.npz'
assert mlp_info['predictions_sha256'] == hashlib.sha256(saved_path.read_bytes()).hexdigest()
with np.load(saved_path, allow_pickle=False) as saved:
    np.testing.assert_array_equal(saved['test_indices'], np.arange(len(test)))
    np.testing.assert_array_equal(saved['truth'], test_family.to_numpy())
    mlp_alarms = saved['predicted'] != 'Normal'

normal = test['attack_label'].eq('normal').to_numpy()
seen = (~normal) & test['attack_label'].isin(train['attack_label']).to_numpy()
unseen = ~normal & ~seen
assert (normal.sum(), seen.sum(), unseen.sum()) == (9711, 9083, 3750)
assert test.loc[unseen, 'attack_label'].nunique() == 17
assert mlp_alarms[normal].sum() == 711 and mlp_alarms[unseen].sum() == 1826

run_labels = ['MLP', *[f'DLHA seed {seed}' for seed in seeds]]
all_alarms = [mlp_alarms, *predictions]
groups = {'visti': seen, 'assenti': unseen, 'falsi_allarmi': normal}
metrics = pd.DataFrame([
    {'modello': model, **{f'{group}_n': int(alarms[mask].sum()) for group, mask in groups.items()},
     **{group: float(alarms[mask].mean()) for group, mask in groups.items()}}
    for model, alarms in zip(run_labels, all_alarms)
]).set_index('modello')
summary = pd.DataFrame([metrics.iloc[0], metrics.iloc[1:].mean()], index=['MLP', 'DLHA (media)'])
display(metrics.round(4))
display(summary.round(4))

## 5. Dettaglio per tipo e per famiglia vera

I tipi assenti restano gli stessi con entrambe le mappature. Per confrontare
anche la figura 8 del paper aggiungiamo le famiglie con la mappa storica, che
riproduce i suoi denominatori. I risultati indicano la quota segnalata della
famiglia vera, perché la cascata restituisce una decisione binaria.

In [ ]:
detail = []
family_maps = {'Divekar': test_family, 'storica (denominatori del paper)': test['attack_label'].map(ATTACK_TO_FAMILY)}
subtype_counts = test.loc[unseen, 'attack_label'].value_counts()
for model, alarms in zip(run_labels, all_alarms):
    for mapping, family_labels in family_maps.items():
        for family in ['DoS', 'Probe', 'R2L', 'U2R']:
            mask = family_labels.eq(family).to_numpy()
            for subset, subset_mask in [('tutti', ~normal), ('assenti', unseen)]:
                selected = mask & subset_mask
                detail.append((model, mapping, family, subset, int(selected.sum()), int(alarms[selected].sum())))
    for attack in subtype_counts.index:
        selected = test['attack_label'].eq(attack).to_numpy()
        detail.append((model, 'nome attacco', attack, 'assenti', int(selected.sum()), int(alarms[selected].sum())))
detail = pd.DataFrame(detail, columns=['modello','mappa','gruppo','sottoinsieme','casi','rilevati'])
detail['rilevamento'] = detail['rilevati'] / detail['casi']
detail_mean = detail[detail['modello'].ne('MLP')].groupby(['mappa','gruppo','sottoinsieme','casi'], as_index=False)['rilevamento'].mean()
display(detail_mean[detail_mean['mappa'].eq('nome attacco')].sort_values('casi', ascending=False).round(4))

## 6. Grafici, confronto con il paper e salvataggio

Il paper rileva 3.411 dei 3.750 casi assenti. Sui falsi allarmi riporta 11,82%,
mentre dai conteggi e dalla precisione si ricava circa 16,5% sui normali.
Per questo mostriamo entrambe le indicazioni nella tabella di confronto e
nei grafici riportiamo le misure calcolate dalle nostre predizioni.

In [ ]:
OUTPUT = ROOT / 'artifacts/dlha_official_test'
ASSETS = ROOT / 'docs/assets/dlha_official_test'
OUTPUT.mkdir(parents=True, exist_ok=True)
ASSETS.mkdir(parents=True, exist_ok=True)

fig, axes = plt.subplots(1, 2, figsize=(11.5, 4.2), layout='constrained')
for ax, columns, title in zip(axes, [['visti','assenti'], ['falsi_allarmi']],
                              ['Attacchi rilevati', 'Connessioni normali segnalate']):
    labels = ['Tipi visti', 'Tipi assenti'] if len(columns) == 2 else ['Falsi allarmi']
    x = np.arange(len(columns))
    for idx, (model, color) in enumerate(zip(summary.index, ['#347fb5','#ed912a'])):
        bars = ax.bar(x + (idx - 0.5)*0.32, summary.loc[model, columns].to_numpy()*100,
                      width=0.32, label=model, color=color)
        ax.bar_label(bars, fmt='%.1f%%', padding=3)
    ax.set(xticks=x, xticklabels=labels, ylim=(0,105), ylabel='Percentuale del gruppo', title=title)
    if len(columns) == 1:
        ax.legend(loc='upper right')
fig.savefig(ASSETS / 'dlha_comparison.png', dpi=180)
plt.show()

binary_matrices = np.array([confusion_matrix(~normal, alarms, labels=[False, True], normalize='true') for alarms in all_alarms])
fig, axes = plt.subplots(1, 2, figsize=(10, 4), layout='constrained')
for ax, model, matrix in zip(axes, summary.index, [binary_matrices[0], binary_matrices[1:].mean(axis=0)]):
    ConfusionMatrixDisplay(matrix, display_labels=['Normale','Attacco']).plot(
        ax=ax, cmap='Blues', values_format='.2%', colorbar=False, im_kw={'vmin': 0, 'vmax': 1})
    ax.set(title=model, xlabel='Predetto', ylabel='Reale')
fig.savefig(ASSETS / 'dlha_confusion.png', dpi=180)
plt.show()

metrics.to_csv(OUTPUT / 'runs.csv')
summary.to_csv(OUTPUT / 'comparison.csv')
detail.to_csv(OUTPUT / 'detection_detail.csv', index=False)
detail_mean.to_csv(OUTPUT / 'dlha_mean_detail.csv', index=False)
np.savez_compressed(OUTPUT / 'predictions.npz', test_indices=np.arange(len(test)), truth=~normal,
                    first_stage=first_stage, dlha=np.stack(predictions), mlp=mlp_alarms,
                    svm_normal_train_indices=np.stack(sampled_indices), seeds=seeds,
                    normalized_confusion=binary_matrices)
paper_values = {'visti': 8539/9083, 'assenti': 3411/3750,
                'falsi_allarmi_pubblicato': 0.1182,
                'falsi_allarmi_stimato': (11950/0.8817 - 11950)/9711}
display(pd.DataFrame([paper_values], index=['DLHA (paper)']).round(4))
(OUTPUT / 'results.json').write_text(json.dumps({
    'protocol_sha256': mlp_info['protocol_sha256'],
    'mlp_predictions_sha256': mlp_info['predictions_sha256'],
    'seeds': seeds, 'nb_rows': len(nb_features), 'svm_transformation_rows': len(svm_features),
    'svm_fit_rows_per_seed': len(sample), 'test_rows_to_svm': int(to_svm.sum()),
    'pca_components': [nb_features.shape[1], svm_features.shape[1]],
    'comparison': summary.to_dict(orient='index'), 'paper': paper_values,
}, indent=2), encoding='utf-8')